# Rossmann Store Sales — Multi-Month Model Validation

## Goal

Evaluate the selected model from Notebook 05 across multiple monthly validation periods rather than relying only on its July 2015 performance.

The objective is to compare the selected CatBoost model with the historical baseline using a chronological validation strategy.

Each validation month uses only observations from dates before the beginning of that month.

These experiments provide additional development validation evidence, rather than a final evaluation on a completely untouched test dataset.

## 1. Load Data
Load the cleaned sales and store data.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
candidate_roots = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "data" / "processed").exists()),
    Path.cwd().parent,
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH = PROCESSED_DATA_DIR / "train_clean.parquet"
STORE_PATH = PROCESSED_DATA_DIR / "store_clean.parquet"

assert TRAIN_PATH.exists(), f"File not found: {TRAIN_PATH}"
assert STORE_PATH.exists(), f"File not found: {STORE_PATH}"

df = pd.read_parquet(TRAIN_PATH)
store_df = pd.read_parquet(STORE_PATH)
df["Date"] = pd.to_datetime(df["Date"])

print("Training data shape:", df.shape)
print("Store metadata shape:", store_df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

Training data shape: (1017209, 9)
Store metadata shape: (1115, 10)
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00


## 2. Build Model Features
Use the same Competition and Historical Sales features as the selected model in Notebook 05.


In [ ]:
all_open = df.loc[df["Open"] == 1].copy()

metadata_columns = [
    "Store",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
]

all_open = all_open.merge(
    store_df[metadata_columns],
    on="Store",
    how="left",
    validate="many_to_one",
)

comp_year = all_open["CompetitionOpenSinceYear"].astype("float64")
comp_month = all_open["CompetitionOpenSinceMonth"].astype("float64")

known_competition_date = comp_year.notna() & comp_month.notna() & (comp_year != 1900)

months_since_opening = (
    (all_open["Date"].dt.year - comp_year) * 12 + all_open["Date"].dt.month - comp_month
)

all_open["CompetitionDateKnown"] = known_competition_date.astype("int8")
all_open["CompetitionActive"] = (
    (months_since_opening >= 0).astype("float64").where(known_competition_date)
)
all_open["CompetitionAgeMonths"] = months_since_opening.clip(lower=0).where(
    known_competition_date
)

assert len(all_open) == int((df["Open"] == 1).sum())
assert all_open.duplicated(["Store", "Date"]).sum() == 0

Historical features use open-store sales from before each forecast month. Sales from the forecast month are not used.


In [ ]:
def add_monthly_historical_sales_features(target, history):
    target_features = target.copy()
    target_features["Date"] = pd.to_datetime(target_features["Date"])
    target_features["_row_position"] = np.arange(len(target_features))
    target_features["_forecast_month"] = (
        target_features["Date"].dt.to_period("M").dt.to_timestamp()
    )

    history_open = history.loc[
        history["Open"] == 1,
        ["Store", "Date", "DayOfWeek", "Sales"],
    ].copy()
    history_open["Date"] = pd.to_datetime(history_open["Date"])

    feature_parts = []

    for forecast_month, month_target in target_features.groupby(
        "_forecast_month",
        sort=True,
    ):
        month_target = month_target.copy()
        available_history = history_open.loc[history_open["Date"] < forecast_month]

        recent_30_days = available_history.loc[
            available_history["Date"] >= forecast_month - pd.Timedelta(days=30)
        ]
        recent_90_days = available_history.loc[
            available_history["Date"] >= forecast_month - pd.Timedelta(days=90)
        ]

        mean_30_days = recent_30_days.groupby("Store")["Sales"].mean()
        mean_90_days = recent_90_days.groupby("Store")["Sales"].mean()
        mean_store_weekday = available_history.groupby(["Store", "DayOfWeek"])[
            "Sales"
        ].mean()

        month_target["Sales_Mean_30D"] = month_target["Store"].map(mean_30_days)
        month_target["Sales_Mean_90D"] = month_target["Store"].map(mean_90_days)

        target_keys = pd.MultiIndex.from_frame(month_target[["Store", "DayOfWeek"]])
        month_target["Sales_Mean_Store_DOW"] = mean_store_weekday.reindex(
            target_keys
        ).to_numpy()

        feature_parts.append(month_target)

    return (
        pd.concat(feature_parts)
        .sort_values("_row_position")
        .drop(columns=["_row_position", "_forecast_month"])
    )

In [5]:
feature_data = add_monthly_historical_sales_features(all_open, all_open)

model_features = [
    "Store",
    "DayOfWeek",
    "Promo",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "CompetitionDateKnown",
    "CompetitionActive",
    "CompetitionAgeMonths",
    "Sales_Mean_30D",
    "Sales_Mean_90D",
    "Sales_Mean_Store_DOW",
]

categorical_features = [
    "Store",
    "DayOfWeek",
    "StoreType",
    "Assortment",
]

catboost_params = {
    "iterations": 300,
    "depth": 6,
    "learning_rate": 0.1,
    "loss_function": "RMSE",
    "random_seed": 42,
    "verbose": False,
}

print("Model features:", len(model_features))
print("Categorical features:", categorical_features)
print("Duplicate Store-Date:", feature_data.duplicated(["Store", "Date"]).sum())

Model features: 12
Categorical features: ['Store', 'DayOfWeek', 'StoreType', 'Assortment']
Duplicate Store-Date: 0


### Feature Table Validation

The final modeling dataset contains 12 features:

- 4 categorical features
- 8 numerical or binary features

The categorical features are:

- `Store`
- `DayOfWeek`
- `StoreType`
- `Assortment`

The feature construction process preserves the original Store-Date structure, and no duplicate Store-Date combinations were found.

Historical sales features are calculated using observations available before the beginning of each forecast month.

This ensures that sales from the forecast month are not used to construct that month's historical features.

The resulting feature table is ready for monthly chronological validation.

## 3. Monthly Validation
Use April, May, June, and July 2015 as validation months. Each model is trained only on dates before its validation month.


In [6]:
validation_months = pd.to_datetime(
    ["2015-04-01", "2015-05-01", "2015-06-01", "2015-07-01"]
)

baseline_groups = ["Store", "DayOfWeek", "Promo"]
fold_results = []
all_actual = []
all_baseline_predictions = []
all_catboost_predictions = []

for start_date in validation_months:
    end_date = start_date + pd.offsets.MonthBegin(1)

    fold_train = feature_data.loc[feature_data["Date"] < start_date].copy()
    fold_valid = feature_data.loc[
        (feature_data["Date"] >= start_date) & (feature_data["Date"] < end_date)
    ].copy()

    if fold_train.empty or fold_valid.empty:
        raise ValueError(f"Empty training or validation data for {start_date:%Y-%m}")

    if fold_train["Date"].max() >= fold_valid["Date"].min():
        raise ValueError("Chronological split is invalid.")

    X_train = fold_train[model_features].copy()
    X_valid = fold_valid[model_features].copy()

    for column in categorical_features:
        X_train[column] = X_train[column].astype(str)
        X_valid[column] = X_valid[column].astype(str)

    model = CatBoostRegressor(
        **catboost_params,
        cat_features=categorical_features,
    )
    model.fit(X_train, fold_train["Sales"])
    catboost_predictions = model.predict(X_valid)

    group_means = fold_train.groupby(baseline_groups)["Sales"].mean()
    validation_keys = pd.MultiIndex.from_frame(fold_valid[baseline_groups])
    baseline_predictions = pd.Series(
        group_means.reindex(validation_keys).to_numpy(),
        index=fold_valid.index,
        dtype="float64",
    )

    store_means = fold_train.groupby("Store")["Sales"].mean()
    baseline_predictions = baseline_predictions.fillna(
        fold_valid["Store"].map(store_means)
    )

    if baseline_predictions.isna().any():
        raise ValueError("Some validation rows do not have a baseline prediction.")

    actual = fold_valid["Sales"].to_numpy()
    baseline_values = baseline_predictions.to_numpy()

    fold_results.append(
        {
            "Month": start_date.strftime("%Y-%m"),
            "Training_Rows": len(fold_train),
            "Validation_Rows": len(fold_valid),
            "Baseline_MAE": mean_absolute_error(actual, baseline_values),
            "CatBoost_MAE": mean_absolute_error(actual, catboost_predictions),
            "Baseline_RMSE": mean_squared_error(actual, baseline_values) ** 0.5,
            "CatBoost_RMSE": mean_squared_error(actual, catboost_predictions) ** 0.5,
        }
    )

    all_actual.append(actual)
    all_baseline_predictions.append(baseline_values)
    all_catboost_predictions.append(catboost_predictions)

fold_results = pd.DataFrame(fold_results)
fold_results["MAE_Improvement_Pct"] = (
    (fold_results["Baseline_MAE"] - fold_results["CatBoost_MAE"])
    / fold_results["Baseline_MAE"]
    * 100
)

display(fold_results.round(2))

,Month,Training_Rows,Validation_Rows,Baseline_MAE,CatBoost_MAE,Baseline_RMSE,CatBoost_RMSE,MAE_Improvement_Pct
0,2015-04,732970,26931,972.65,959.77,1484.59,1408.03,1.32
1,2015-05,759901,25880,843.40,805.88,1190.62,1115.74,4.45
2,2015-06,785781,28423,748.15,713.03,1132.42,1056.65,4.69
3,2015-07,814204,30188,732.18,718.09,1038.49,991.79,1.92


## 4. Overall Results
Compare both models across all four validation months. Positive MAE improvement means CatBoost has lower error.


In [7]:
actual_all = np.concatenate(all_actual)
baseline_all = np.concatenate(all_baseline_predictions)
catboost_all = np.concatenate(all_catboost_predictions)

summary = pd.DataFrame(
    {
        "Model": ["Historical Baseline", "CatBoost + Competition + Historical Sales"],
        "MAE": [
            mean_absolute_error(actual_all, baseline_all),
            mean_absolute_error(actual_all, catboost_all),
        ],
        "RMSE": [
            mean_squared_error(actual_all, baseline_all) ** 0.5,
            mean_squared_error(actual_all, catboost_all) ** 0.5,
        ],
    }
).set_index("Model")

display(summary.round(2))

better_months = int((fold_results["CatBoost_MAE"] < fold_results["Baseline_MAE"]).sum())

print(
    f"CatBoost has lower MAE in {better_months} "
    f"of {len(fold_results)} validation months."
)

baseline_mae = summary.loc["Historical Baseline", "MAE"]
catboost_mae = summary.loc["CatBoost + Competition + Historical Sales", "MAE"]
improvement_pct = (baseline_mae - catboost_mae) / baseline_mae * 100
print(f"Combined MAE improvement: {improvement_pct:.2f}%")

if catboost_mae < baseline_mae:
    print("CatBoost has lower MAE across the combined validation period.")
else:
    print(
        "The historical baseline has lower MAE across the combined validation period."
    )

,MAE,RMSE
Model,,
Historical Baseline,820.21,1217.09
CatBoost + Competition + Historical Sales,795.60,1148.76


CatBoost has lower MAE in 4 of 4 validation months.
Combined MAE improvement: 3.00%
CatBoost has lower MAE across the combined validation period.


## 5. Conclusion

The selected CatBoost model from Notebook 05 was evaluated across four consecutive monthly validation periods from April to July 2015.

The model was compared against the historical baseline using a chronological validation strategy.

### Monthly Results

CatBoost achieved lower MAE than the historical baseline in all four validation months.

| Month | Baseline MAE | CatBoost MAE | MAE Improvement |
|---|---:|---:|---:|
| April 2015 | 972.65 | 959.77 | 1.32% |
| May 2015 | 843.40 | 805.88 | 4.45% |
| June 2015 | 748.15 | 713.03 | 4.69% |
| July 2015 | 732.18 | 718.09 | 1.92% |

The improvement is consistent across all four months, although its magnitude varies.

### Combined Results

When all validation observations are combined, the results are:

| Metric | Historical Baseline | Selected CatBoost |
|---|---:|---:|
| MAE | 820.21 | 795.60 |
| RMSE | 1,217.09 | 1,148.76 |

The selected CatBoost model achieves:

- A combined MAE improvement of approximately 3.00%.
- A lower combined RMSE of 1,148.76 compared with 1,217.09 for the baseline.
- Lower MAE in all four monthly validation periods.

### Interpretation

The results suggest that adding store characteristics, competition information, and historical sales features provides useful predictive information beyond the historical baseline.

The consistent improvement across the four evaluated months is encouraging.

However, the improvement should be interpreted carefully because the validation periods were used during the broader model development process.

### Limitations and Next Steps

These monthly validation periods are development validation data, not a final untouched test set.

A separate future time period should be reserved for a final evaluation after all feature and model selection decisions have been completed.

The original Rossmann Kaggle competition uses RMSPE as its official evaluation metric, whereas this project currently focuses on MAE and RMSE.

Therefore, the current results should not be directly compared with Kaggle leaderboard scores.

If competition-aligned evaluation is desired, RMSPE should also be calculated using the official competition's evaluation rules.

Reference: [Rossmann Store Sales - Official Kaggle Competition](https://www.kaggle.com/competitions/rossmann-store-sales)

Overall, the selected CatBoost model outperforms the historical baseline across the evaluated months and provides a promising result for further experimentation.